In [ ]:
# document 정보 최종파일(었_결합_로그오즈비, 등급, 성향 포함), 

# ..\csv\original_csv\세종문어_document_정보_ver.1.2.csv로 저장했습니다.
#완료 - 20260323_01-18-12 file 정보와 합함, 컬럼명 조정

# file 정보 최종파일(었_결합_로그오즈비, 등급, 성향 포함)
# ..\csv\original_csv\세종문어_file_정보_ver.1.1.csv로 저장했습니다.
#완료 - 20260323_01-18-12

In [ ]:
#각종 함수 선언 셀, process_dataframe, generate_pivot_statistics, expand_window
import pandas as pd
import numpy as np
import gc
from pandas.api.types import is_categorical_dtype
import re
from pathlib import Path
from datetime import datetime

import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))
from stats.v_affix_attach import v_affix_attach, add_v_no_by_merge_and_fallback
from stats.en_no_fix import fix_en_number_with_merge, make_en_no_sub #EN_No 수정


In [ ]:
from pathlib import Path
from datetime import datetime

#대상 파일 읽어오기.
CSV_PATH = Path(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.7.csv")
df_word = pd.read_csv(CSV_PATH)
print(f"전체 {len(df_word):,}행, now: {datetime.now().strftime("%Y.%m.%d_%H:%M:%S")}")

#문장 파일 읽기
SEN_CSV = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.7_sen.csv"     
df_sen = pd.read_csv(SEN_CSV, low_memory=False)
# 컬럼명 변경
df_sen = df_sen.rename(columns={
        "last_EN_No": "sentence_f_EN_No",
        "last_EN_No": "sentence_f_EN_No",
        "last_EN_in_quote": "sentence_f_EN_in_quote",
        "last_EN_label": "sentence_f_EN_label"
    })

#document 파일 읽기
DOCU_CSV = r"..\csv\original_csv\세종문어_document_정보.csv"
df_docu = pd.read_csv(DOCU_CSV, low_memory=False)
DOCU_CSV2 = r"..\csv\original_csv\세종문어_document_었결합률.csv"
df_docu2 = pd.read_csv(DOCU_CSV2, low_memory=False)
#file 파일 읽기
FILE_CSV = r"..\csv\original_csv\세종문어_file_정보.csv"
df_file = pd.read_csv(FILE_CSV, low_memory=False)

print(f"now: {datetime.now().strftime("%Y.%m.%d_%H:%M:%S")}")
print(f"df_file: {len(df_file):,}행, df_docu: {len(df_docu):,}행, df_docu2: {len(df_docu2):,}행, df_sen: {len(df_sen):,}행, df: {len(df_word):,}행")

전체 12,973,652행, now: 2026.07.01_21:45:09
now: 2026.07.01_21:45:14
df_file: 369행, df_docu: 33,155행, df_sen: 1,048,127행, df: 12,973,652행


In [5]:
df_docu.columns

Index(['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목',
       '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3',
       '분류기호4', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio'],
      dtype='object')

In [7]:
df_sen.columns

Index(['file_id', 'docu_id', 'sen_id', 'file_sen_num', 'sen_num',
       'rev_sen_num', 'file_name', 'docu_no', 'sentence_form', 'speaker',
       'sen_len', 'quote_type', 'sent_has_E', 'sentence_f_EN_No',
       'last_EN_label', 'sentence_f_EN_in_quote', 'sentence_f_EP_form',
       'quote_count', 'quote_positions', 'quote_group_id', 'quote_span_len',
       'long_quote', 'quote_start_sen_id', 'quote_end_sen_id', 'span_wrap_ok',
       'span_type', 'passage_type', 'review_flag', 'review_note',
       'quote_type_fix', 'modified_at', 'quote_end_offset'],
      dtype='object')

In [4]:
#데이터프레임의 결측치 확인 함수
def check_na_df(df, cols):
    result = []
    
    for col in cols:
        total = len(df)
        na = df[col].isna().sum()
        not_na = df[col].count()
        
        result.append({
            "column": col,
            "total": total,
            "na": na,
            "not_na": not_na,
            "na_ratio": na / total
        })
    
    return pd.DataFrame(result)

check_na_df(df_docu, ['매체', 'file_id', 'docu_id'])

,column,total,na,not_na,na_ratio
0,매체,33155,0,33155,0.0
1,file_id,33155,0,33155,0.0
2,docu_id,33155,0,33155,0.0


In [ ]:
## df_docu에 docu_id가 비어있어 정보 공백이 생긴 것 처리
import pandas as pd
df = df_word.copy()
# ---------- 0) 키 컬럼 정리(가짜 불일치 방지) ----------
for _df in (df, df_docu, df_file):
    for k in ["file_id", "docu_id"]:
        if k in _df.columns:
            _df[k] = _df[k].astype(str).str.strip()

# ---------- 1) df에서 docu_id 목록(=문서 목록) 만들기 ----------
# df는 어절 단위라고 했으니 docu_id별로 "대표 1행"을 뽑아 docu 레벨 테이블로 만든다
need_cols_from_df = [c for c in ["docu_id", "file_id", "category", '매체', '내용', '파일제목', '저자', '출판사', '출판연도', 'head', 'docu_lenth', 'docu_lenth(onlyEF)', 'dominant_EN_No', 'dominant_count', 'dominant_ratio']
                     if c in df.columns]

docu_from_df = (
    df[need_cols_from_df]
    .dropna(subset=["docu_id"])
    .drop_duplicates("docu_id")
)

# ---------- 2) df_docu에 누락된 docu 추가(행 보충) ----------
# df_docu에 없는 docu_id를 추가하되, df_docu 컬럼 구조를 유지하면서 붙인다
docu_ids_docu = set(df_docu["docu_id"].dropna().unique()) if "docu_id" in df_docu.columns else set()
docu_ids_df   = set(docu_from_df["docu_id"].dropna().unique())

missing_docu_ids = sorted(docu_ids_df - docu_ids_docu)

if missing_docu_ids:
    # df_docu 컬럼에 맞춰 빈 행 프레임 만든 뒤, df에서 가져올 수 있는 컬럼만 채워 넣기
    add_rows = pd.DataFrame({"docu_id": missing_docu_ids})
    add_rows = add_rows.merge(docu_from_df, on="docu_id", how="left")

    # df_docu에 없는 컬럼은 제거, df_docu에 있는 컬럼은 맞춰서 생성
    for c in df_docu.columns:
        if c not in add_rows.columns:
            add_rows[c] = pd.NA
    add_rows = add_rows[df_docu.columns]

    df_docu = pd.concat([df_docu, add_rows], ignore_index=True)

# ---------- 3) df_docu의 빈 값만 df에서 채우기(docu_id 기준) ----------
# df_docu에 이미 있는 docu도 "메타가 비어있다면" df에서 가져와서 채운다
fill_cols_docu = [c for c in docu_from_df.columns if c not in ["docu_id"]]

tmp = df_docu.merge(docu_from_df[["docu_id"] + fill_cols_docu],
                    on="docu_id", how="left", suffixes=("", "_df"))

def is_empty(s: pd.Series) -> pd.Series:
    if pd.api.types.is_string_dtype(s):
        return s.isna() | (s.astype("string").str.strip() == "")
    return s.isna()

for c in fill_cols_docu:
    c_df = f"{c}_df"
    if c in tmp.columns and c_df in tmp.columns:
        m = is_empty(tmp[c])
        tmp.loc[m, c] = tmp.loc[m, c_df]
        tmp.drop(columns=[c_df], inplace=True)

df_docu = tmp

# ---------- 4) 마지막으로 file_id 기준으로 df_file에서 빈 값만 채우기 ----------
df_file_u = df_file.drop_duplicates("file_id")

fill_cols_file = [c for c in df_file_u.columns if c != "file_id"]

tmp = df_docu.merge(df_file_u[["file_id"] + fill_cols_file],
                    on="file_id", how="left", suffixes=("", "_file"))

for c in fill_cols_file:
    c_file = f"{c}_file"
    if c in tmp.columns and c_file in tmp.columns:
        m = is_empty(tmp[c])
        tmp.loc[m, c] = tmp.loc[m, c_file]
        tmp.drop(columns=[c_file], inplace=True)

df_docu = tmp

# ---------- 5) 결과 점검 ----------
print("df에서 docu_id 총개수:", len(docu_ids_df))
print("df_docu에서 docu_id 총개수:", df_docu["docu_id"].nunique())
print("df_docu에 새로 추가된 docu_id 개수:", len(missing_docu_ids))

# (선택) 아직도 비어있는 docu가 있는지
# 예: file_id가 비어있으면 df_file에서 채울 수가 없음
if "file_id" in df_docu.columns:
    print("file_id 비어있는 docu 행 수:", df_docu["file_id"].isna().sum())

In [ ]:
# 필요시 저장
df_docu.to_csv(DOCU_CSV, index=False, encoding= "utf-8")
print(f"{DOCU_CSV}로 저장했습니다.")

In [ ]:
## df_docu와 df_file의 file_id 불일치 점검

# 0) (선택) 타입/공백 차이로 인한 "가짜 불일치" 방지
df_docu["file_id"] = df_docu["file_id"].astype(str).str.strip()
df_file["file_id"] = df_file["file_id"].astype(str).str.strip()

# 1) 각각의 file_id 집합 만들기
docu_ids = set(df_docu["file_id"].dropna().unique())
file_ids = set(df_file["file_id"].dropna().unique())

# 2) 서로에만 있는 file_id 찾기
only_in_docu = sorted(docu_ids - file_ids)   # df_docu에만 있음
only_in_file = sorted(file_ids - docu_ids)   # df_file에만 있음

print("df_docu에만 있는 file_id 개수:", len(only_in_docu))
print("df_file에만 있는 file_id 개수:", len(only_in_file))

# 3) 샘플로 확인
print("df_docu에만 있는 예시:", only_in_docu[:20])
print("df_file에만 있는 예시:", only_in_file[:20])

In [ ]:
df.loc[df["file_id"] == "GO0097", "category"] = "상상-아동"
df.loc[df["file_id"] == "BZ0074", "category"] = "총류"
df.loc[df["file_id"] == "GO0341", "category"] = "체험기술"
df.loc[df["file_id"] == "HO0102", "category"] = "체험기술"
df.loc[df["file_id"] == "HO0115", "category"] = "상상-일반" 


In [21]:
#값 교체
df_docu[["category"]] = df_docu[["category"]].replace('보도·해설', '보도해설')
df_docu[["category"]] = df_docu[["category"]].replace('상상-일반', '허구일반')
df_docu[["category"]] = df_docu[["category"]].replace('상상-아동', '허구아동')

In [30]:
df_docu2['었_결합_로그오즈비'].describe()

count    33155.000000
mean        -0.373441
std          1.718237
min         -5.579806
25%         -1.666536
50%         -0.485007
75%          1.019081
max          4.436889
Name: 었_결합_로그오즈비, dtype: float64

In [ ]:
def classify_log_odds_5(x):
    if pd.isna(x):
        return None
    elif x <= -1.5:
        return 1   # 강한 회피
    elif x <= -0.5:
        return 2   # 약한 회피
    elif x < 0.5:
        return 3   # 중립
    elif x < 1.5:
        return 4   # 약한 선호
    else:
        return 5   # 강한 선호

df_docu2['었_결합_등급'] = df_docu2['었_결합_로그오즈비'].apply(classify_log_odds_5)

df_docu2['었_결합_성향'] = df_docu2['었_결합_등급'].map({
    1: '회피',
    2: '회피',
    3: '중립',
    4: '선호',
    5: '선호'
})
'''문서별 ‘었’ 결합 경향은 로그오즈비(log odds ratio)를 기준으로 산출하였다. 
로그오즈비는 0을 기준으로 양수는 ‘었’ 결합의 선호, 음수는 회피를 의미하며, 절댓값이 클수록 그 경향이 강해진다.

본 자료의 로그오즈비 분포를 살펴보면, 전체 33,155개 문서의 중앙값은 -0.485로 나타나 약한 회피 경향이 전반적으로 우세하였으며, 제1사분위수는 -1.67, 제3사분위수는 1.02로 나타나 대부분의 문서가 이 범위 내에 분포하였다.
'''
'''방법론/분류 기준

문서 단위와 파일 단위의 로그오즈비 분포는 분산의 크기에서 차이를 보였으나, 두 단위의 결과를 직접 비교하기 위해 동일한 기준을 적용하였다. 로그오즈비 0은 ‘-었-’ 결합에 대한 상대적 편향이 없는 상태를 의미하므로, 본 연구에서는 -0.5~0.5 구간을 중립으로 설정하였다. 또한 이 범위를 벗어난 값에 대해서는 ±1.5를 기준으로 약한 경향과 강한 경향을 구분하여, 강한 회피(≤ -1.5), 약한 회피(-1.5 초과 -0.5 이하), 중립(-0.5 초과 0.5 미만), 약한 선호(0.5 이상 1.5 미만), 강한 선호(≥ 1.5)의 5단계로 분류하였다.

결과 해석

문서 단위의 로그오즈비 분포는 파일 단위보다 분산이 더 크게 나타났으며, 극단적인 회피 및 선호 구간에 속하는 사례도 더 많이 확인되었다. 반면 파일 단위에서는 값이 상대적으로 중심부에 집중되는 경향이 나타났다. 이는 상위 단위인 파일 내부에 서로 다른 성향의 문서들이 함께 포함되면서 평균화 효과가 발생하기 때문으로 해석할 수 있다. 따라서 ‘-었-’ 결합 경향은 파일 단위에서도 일정한 방향성을 보이지만, 실제 담화적 성향은 문서 단위에서 더 선명하게 드러난다고 볼 수 있다.'''

'''문서 단위와 파일 단위의 분포 폭에는 차이가 있었으나, 두 단위의 결과를 직접 비교하기 위해 동일한 기준을 적용하였다. 중립 구간은 로그오즈비 0을 중심으로 한 -0.5~0.5로 설정하였고, 그 바깥을 약한 경향과 강한 경향으로 구분하기 위해 ±1.5를 경계값으로 사용하였다.'''
'''

'문서별 ‘었’ 결합 경향은 로그오즈비(log odds ratio)를 기준으로 산출하였다. \n로그오즈비는 0을 기준으로 양수는 ‘었’ 결합의 선호, 음수는 회피를 의미하며, 절댓값이 클수록 그 경향이 강해진다.\n\n본 자료의 로그오즈비 분포를 살펴보면, 전체 33,155개 문서의 중앙값은 -0.485로 나타나 약한 회피 경향이 전반적으로 우세하였으며, 제1사분위수는 -1.67, 제3사분위수는 1.02로 나타나 대부분의 문서가 이 범위 내에 분포하였다.\n\n이에 따라 본 연구에서는 로그오즈비의 해석 가능성과 분포 특성을 함께 고려하여 구간을 설정하였다. \n우선 로그오즈비 0을 중심으로 한 -0.5~0.5 구간을 ‘중립’으로 설정하여, ‘었’ 결합 여부에 뚜렷한 편향이 없는 문서를 구분하였다. \n그 바깥 구간은 단계적으로 확장하여 -2.0 이하를 ‘강한 회피’, -2.0 초과 -0.5 이하를 ‘약한 회피’, 0.5 이상 1.5 미만을 ‘약한 선호’, 1.5 이상을 ‘강한 선호’로 구분하였다.\n\n또한 해석의 단순화를 위해 이를 ‘회피–중립–선호’의 3범주로 재분류하여 분석을 병행하였다.'

In [39]:
df_docu.columns

Index(['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목',
       '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3',
       '분류기호4', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio'],
      dtype='object')

In [40]:
# ---
# document 정보에서 필요한 컬럼 병합
# ---
def safe_merge(df, df_add, key, cols):
    cols_to_use = [key] + [c for c in cols if c not in df.columns]
    return df.merge(df_add[cols_to_use], on=key, how="left")

df = safe_merge(df_docu, df_docu2, "docu_id", df_docu2.columns.to_list())
from datetime import datetime
stamp = datetime.now().strftime('%Y.%m.%d_%H:%M:%S')
print(f"merge 완료 - {stamp}")
df.columns

merge 완료 - 2026.03.23_01:45:28


Index(['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목',
       '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3',
       '분류기호4', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'sent_count',
       'head_count', 'body_count', 'body_has_E_count', 'body_not_quote_count',
       'body_not_quote_and_었_count', '었_결합_오즈비', '었_결합_로그오즈비', '었_결합_등급',
       '었_결합_성향'],
      dtype='object')

In [41]:
# 저장하기
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")

output_path = r"../csv/original_csv/세종문어_document_정보_ver.1.1.csv"
df.to_csv(output_path, index=False, encoding= "utf-8")
print(f"{output_path}로 저장했습니다.")
print(f"완료 - {stamp}")

../csv/original_csv/세종문어_document_정보_ver.1.1.csv로 저장했습니다.
완료 - 20260323_01-45-33


In [36]:
#document２ 파일 읽기
FILE_CSV = r"..\csv\original_csv\세종문어_file_정보2.csv"
df_file2 = pd.read_csv(FILE_CSV, low_memory=False)
df_file2.columns

Index(['file_id', 'sent_count', 'head_count', 'body_count', 'body_has_E_count',
       'body_not_quote_count', 'body_not_quote_and_었_count', '었_결합_오즈비',
       '었_결합_로그오즈비'],
      dtype='object')

In [37]:
df_file2['었_결합_로그오즈비'].describe()

count    369.000000
mean      -0.229877
std        1.124935
min       -4.918593
25%       -0.996962
50%       -0.381964
75%        0.585658
max        2.435603
Name: 었_결합_로그오즈비, dtype: float64

In [43]:
df_file2['었_결합_등급'] = df_file2['었_결합_로그오즈비'].apply(classify_log_odds_5)

df_file2['었_결합_성향'] = df_file2['었_결합_등급'].map({
    1: '회피',
    2: '회피',
    3: '중립',
    4: '선호',
    5: '선호'
})
'''파일 단위의 ‘-었-’ 결합 경향은 로그오즈비(log odds ratio)를 기준으로 산출하였다. 
로그오즈비는 0을 중심으로 양수는 ‘-었-’ 결합의 선호, 음수는 회피를 의미하며, 절댓값이 클수록 그 경향이 강해진다.

전체 369개 파일의 로그오즈비 분포를 살펴보면, 평균은 -0.230, 중앙값은 -0.382로 나타나 전반적으로 약한 회피 경향이 확인되었다. 
또한 제1사분위수는 -0.997, 제3사분위수는 0.586으로 나타나 대부분의 파일이 이 범위에 분포하였다.

이에 본 연구에서는 로그오즈비의 해석 가능성과 자료의 분포를 함께 고려하여 구간을 설정하였다. 
우선 로그오즈비 0을 중심으로 한 -0.5~0.5 구간을 ‘중립’으로 설정하여, ‘-었-’ 결합 여부에 뚜렷한 편향이 없는 경우를 구분하였다. 
그 바깥 구간은 단계적으로 확장하여 -1.5 이하를 ‘강한 회피’, -1.5 초과 -0.5 이하를 ‘약한 회피’, 0.5 이상 1.5 미만을 ‘약한 선호’, 1.5 이상을 ‘강한 선호’로 구분하였다.

또한 해석의 명확성을 위해 이 5단계 구분을 ‘회피–중립–선호’의 3범주로 재분류하여 분석을 병행하였다.'''


'파일 단위의 ‘-었-’ 결합 경향은 로그오즈비(log odds ratio)를 기준으로 산출하였다. \n로그오즈비는 0을 중심으로 양수는 ‘-었-’ 결합의 선호, 음수는 회피를 의미하며, 절댓값이 클수록 그 경향이 강해진다.\n\n전체 369개 파일의 로그오즈비 분포를 살펴보면, 평균은 -0.230, 중앙값은 -0.382로 나타나 전반적으로 약한 회피 경향이 확인되었다. \n또한 제1사분위수는 -0.997, 제3사분위수는 0.586으로 나타나 대부분의 파일이 이 범위에 분포하였다.\n\n이에 본 연구에서는 로그오즈비의 해석 가능성과 자료의 분포를 함께 고려하여 구간을 설정하였다. \n우선 로그오즈비 0을 중심으로 한 -0.5~0.5 구간을 ‘중립’으로 설정하여, ‘-었-’ 결합 여부에 뚜렷한 편향이 없는 경우를 구분하였다. \n그 바깥 구간은 단계적으로 확장하여 -1.5 이하를 ‘강한 회피’, -1.5 초과 -0.5 이하를 ‘약한 회피’, 0.5 이상 1.5 미만을 ‘약한 선호’, 1.5 이상을 ‘강한 선호’로 구분하였다.\n\n또한 해석의 명확성을 위해 이 5단계 구분을 ‘회피–중립–선호’의 3범주로 재분류하여 분석을 병행하였다.'

In [ ]:
# ---
# file 정보에서 필요한 컬럼 병합
# ---
def safe_merge(df, df_add, key, cols):
    cols_to_use = [key] + [c for c in cols if c not in df.columns]
    return df.merge(df_add[cols_to_use], on=key, how="left")

df = safe_merge(df_file, df_file2, "file_id", df_file2.columns.to_list())
from datetime import datetime
stamp = datetime.now().strftime('%Y.%m.%d_%H:%M:%S')
print(f"merge 완료 - {stamp}")
df.columns

merge 완료 - 2026.03.23_01:48:14


Index(['file_id', '매체', '내용', '내용2', '파일제목', '저자', '출판사', '출판연도', '구어/문어',
       '분류기호', '분류기호2', '내용3', '분류기호4', 'sent_count', 'head_count',
       'body_count', 'body_has_E_count', 'body_not_quote_count',
       'body_not_quote_and_었_count', '었_결합_오즈비', '었_결합_로그오즈비', '었_결합_등급',
       '었_결합_성향'],
      dtype='object')

In [45]:
# 저장하기
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")

output_path = r"../csv/original_csv/세종문어_file_정보_ver.1.1.csv"
df.to_csv(output_path, index=False, encoding= "utf-8")
print(f"{output_path}로 저장했습니다.")
print(f"완료 - {stamp}")

../csv/original_csv/세종문어_file_정보_ver.1.1.csv로 저장했습니다.
완료 - 20260323_01-48-45


In [ ]:
#document 파일 읽기
DOCU_CSV = r"..\csv\original_csv\세종문어_document_정보_ver.1.1.csv"
df_docu = pd.read_csv(DOCU_CSV, low_memory=False)

#category2 컬럼 생성 (category + 매체)
df_docu["category2"] = df_docu["category"] + "_" + df_docu["매체"]
cols= ['sen_count', 'sen_count_has_E', 'sen_count_not_quote', 'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No', 'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'sent_count', 'head_count', 'body_count', 'body_has_E_count', 'body_not_quote_count', 'body_not_quote_and_었_count', '었_결합_오즈비', '었_결합_로그오즈비', '었_결합_등급', '었_결합_성향']

df_docu = df_docu.rename(columns={c: f'docu_{c}' for c in cols})

#file 파일 읽기
FILE_CSV = r"..\csv\original_csv\세종문어_file_정보_ver.1.1.csv"
df_file = pd.read_csv(FILE_CSV, low_memory=False)

# 컬럼명 수정
cols = [
    'sent_count', 'head_count', 'body_count', 'body_has_E_count',
    'body_not_quote_count', 'body_not_quote_and_었_count',
    '었_결합_오즈비', '었_결합_로그오즈비',
    '었_결합_등급', '었_결합_성향'
]

df_file = df_file.rename(columns={c: f'file_{c}' for c in cols})

print(f"파일 읽기 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
print(f"df_file: {df_file.columns.tolist()}")
print(f"df_docu: {df_docu.columns.tolist()}")
print(f"df_file: {len(df_file):,}행, df_docu: {len(df_docu):,}행")

In [ ]:
# ---
# document 정보에서 필요한 컬럼 병합
# ---
def safe_merge(df, df_add, key, cols):
    cols_to_use = [key] + [c for c in cols if c not in df.columns]
    return df.merge(df_add[cols_to_use], on=key, how="left")

cols = [c for c in df_file.columns if c != "file_id"]
df_docu = safe_merge(df_docu, df_file, "file_id", cols)

print(f"merge 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

In [ ]:
df_docu.to_csv(r"..\csv\original_csv\세종문어_document_정보_ver.1.2.csv", index=False, encoding="utf-8-sig")